# 01 — Synthetic Customer Data Exploration

This notebook explores the synthetic Takealot-style customer dataset generated by `src/data_generator.py`.
The dataset contains **10 000 customers** spread across three behavioural segments:

| Segment | Share | Spend distribution | Churn behaviour |
|---|---|---|---|
| `high_value` | 20 % | Gamma(α=50, β=10) → mean ≈ R 500/mo | Weibull k=1.2 → increasing hazard, mean ≈ 37 mo |
| `standard` | 50 % | Gamma(α=30, β=5) → mean ≈ R 150/mo | Weibull k=1.0 → constant hazard (exponential), mean ≈ 25 mo |
| `churn_risk` | 30 % | Gamma(α=15, β=3) → mean ≈ R 45/mo | Weibull k=0.8 → decreasing hazard (infant mortality), mean ≈ 11 mo |

**Goal:** Build intuition about spending levels, churn timing, and survival rates before we run any Monte Carlo CLV simulations.

---
## 0 — Imports

In [ ]:
import pathlib

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

# Consistent colours per segment throughout the notebook
SEGMENT_COLOURS = {
    "high_value": "#2196F3",   # blue
    "standard":   "#4CAF50",   # green
    "churn_risk": "#F44336",   # red
}

SEGMENT_ORDER = ["high_value", "standard", "churn_risk"]

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

---
## 1 — Load Data

We read `reports/synthetic_customers.csv`, which was produced by running `src/data_generator.py`.
Each row represents one customer. The key columns we will use throughout are:

- **`segment`** — which of the three behavioural groups this customer belongs to.
- **`monthly_spending`** — how much (in rands) this customer spends per active month.
- **`months_to_churn`** — how many months after acquisition before the customer churns.
  This is drawn from a Weibull distribution, so it is a *continuous* time-to-event variable.
- **`contract_type`** — month-to-month / annual / multi-year.

A quick sanity check confirms row counts match the expected segment proportions.

In [ ]:
DATA_PATH = pathlib.Path("../reports/synthetic_customers.csv")

df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.head()

In [ ]:
# Verify segment proportions match the generator config (20 / 50 / 30 %)
(
    df["segment"]
    .value_counts(normalize=True)
    .rename("proportion")
    .to_frame()
    .style.format("{:.1%}")
)

---
## 2 — Summary Statistics

We calculate **mean**, **median**, and **standard deviation** of `monthly_spending` and `months_to_churn`
for each segment.

**What this tells us:**
- The *mean* is pulled upward by high spenders (the Gamma distribution has a long right tail),
  so the *median* is a more robust central tendency measure for `monthly_spending`.
- The high coefficient of variation (std / mean) in `churn_risk` reflects the Weibull shape k < 1:
  most customers churn very early, but a small tail survive much longer.
- These numbers will anchor our prior distributions when we set up the Monte Carlo simulation.

In [ ]:
def summary_stats(group: pd.DataFrame) -> pd.Series:
    return pd.Series({
        "n": len(group),
        "spend_mean":   group["monthly_spending"].mean(),
        "spend_median": group["monthly_spending"].median(),
        "spend_std":    group["monthly_spending"].std(),
        "spend_cv":     group["monthly_spending"].std() / group["monthly_spending"].mean(),
        "churn_mean":   group["months_to_churn"].mean(),
        "churn_median": group["months_to_churn"].median(),
        "churn_std":    group["months_to_churn"].std(),
    })

stats = (
    df.groupby("segment")
    .apply(summary_stats)
    .loc[SEGMENT_ORDER]  # consistent ordering
    .round(2)
)

stats.style.format({
    "n": "{:,.0f}",
    "spend_mean":   "R {:.2f}",
    "spend_median": "R {:.2f}",
    "spend_std":    "R {:.2f}",
    "spend_cv":     "{:.2f}",
    "churn_mean":   "{:.1f} mo",
    "churn_median": "{:.1f} mo",
    "churn_std":    "{:.1f} mo",
})

---
## 3 — Spending Distribution

We plot **overlapping histograms** of `monthly_spending` for each segment.

**What to look for:**
- The distributions should barely overlap — the three segments were designed with clearly
  separated spend means (≈ R 500, R 150, R 45), confirming the generator worked correctly.
- All three distributions are right-skewed (Gamma family), with the high-value segment
  showing the widest absolute spread.
- This separation means `segment` is a strong feature for any CLV model: knowing a customer's
  segment immediately tells you most of what you need to know about their spend level.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for segment in SEGMENT_ORDER:
    values = df.loc[df["segment"] == segment, "monthly_spending"]
    ax.hist(
        values,
        bins=60,
        alpha=0.55,
        color=SEGMENT_COLOURS[segment],
        label=f"{segment}  (n={len(values):,}, median=R{values.median():.0f})",
        edgecolor="none",
    )

ax.set_xlabel("Monthly Spending (R)", fontsize=12)
ax.set_ylabel("Number of Customers", fontsize=12)
ax.set_title("Spending Distribution by Segment", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R {x:,.0f}"))

plt.tight_layout()
plt.savefig("../reports/figures/spending_distribution.png", bbox_inches="tight")
plt.show()

---
## 4 — Churn Distribution (KDE)

Instead of a histogram we use a **Kernel Density Estimate (KDE)** — a smoothed probability
density curve — so we can see the *shape* of each segment's churn timing without being
distracted by bin-width choices.

**What to look for:**
- **`churn_risk`** peaks sharply near 0 months. This is *infant mortality*: the Weibull
  shape k < 1 means the hazard rate is *highest right at acquisition* and falls over time.
  In practice, these are customers who signed up but never really engaged.
- **`standard`** has k = 1, which gives a memoryless (exponential) distribution — the flat
  density tail means a customer who has survived 12 months has the same churn probability
  as a brand-new one.
- **`high_value`** has k > 1, so the hazard *increases* with tenure — these customers become
  progressively more likely to churn as they age. Their KDE is shifted far to the right,
  meaning they survive much longer on average.

Recognising these patterns matters: a win-back campaign targeting `churn_risk` customers
must fire within the first few months, while retention for `high_value` customers is a
longer-horizon problem.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x_grid = np.linspace(0, 150, 500)

for segment in SEGMENT_ORDER:
    values = df.loc[df["segment"] == segment, "months_to_churn"].values
    kde = gaussian_kde(values, bw_method="scott")
    density = kde(x_grid)

    ax.plot(
        x_grid, density,
        color=SEGMENT_COLOURS[segment],
        linewidth=2.5,
        label=segment,
    )
    ax.fill_between(x_grid, density, alpha=0.15, color=SEGMENT_COLOURS[segment])

    # Mark the median
    median_val = np.median(values)
    ax.axvline(
        median_val,
        color=SEGMENT_COLOURS[segment],
        linestyle="--",
        linewidth=1.2,
        alpha=0.8,
    )

ax.set_xlabel("Months to Churn", fontsize=12)
ax.set_ylabel("Probability Density", fontsize=12)
ax.set_title("Churn Timing Distribution by Segment (KDE)\nDashed lines = segment medians",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.set_xlim(0, 150)

plt.tight_layout()
plt.savefig("../reports/figures/churn_kde.png", bbox_inches="tight")
plt.show()

---
## 5 — Cohort Survival Curves

### What is a survival curve?

A **survival curve** (also called the *survival function* S(t)) answers one question:

> *"What fraction of customers who joined at time 0 are still active at month t?"*

Formally:

$$S(t) = P(T > t)$$

where T is the random variable representing the time-to-churn for a customer drawn from
that segment.

In practice, given a sample, we estimate it empirically:

$$\hat{S}(t) = \frac{\text{number of customers with } months\_to\_churn > t}{\text{total customers in segment}}$$

Every survival curve starts at **S(0) = 1.0** (all customers are active at acquisition)
and is monotonically non-increasing. It eventually approaches 0 once all customers have
churned.

### Why does it matter for CLV modelling?

Customer Lifetime Value is the sum of all future discounted revenue a customer will generate.
If a customer spends R per month and the monthly discount rate is r, then:

$$\text{CLV} = \sum_{t=1}^{\infty} \frac{R \cdot S(t)}{(1+r)^t}$$

The survival curve **directly weights every future cash flow**. Without it, you are
implicitly assuming customers stay forever — wildly overestimating CLV for `churn_risk`
customers who are likely gone within three months.

Three practical uses in this project:
1. **Monte Carlo inputs** — we sample time-to-churn from the fitted Weibull, which is
   exactly the inverse of the survival function.
2. **Segment prioritisation** — `high_value` customers have much higher S(24) than others;
   retention spend there has a longer payback horizon.
3. **Churn intervention timing** — the slope of S(t) is the hazard rate. A steeply
   falling curve tells you *when* to intervene; a flat curve tells you that customers who
   survive this long are unlikely to churn soon.

In [ ]:
months = np.arange(0, 61)  # month 0 to 60

survival_curves = {}

for segment in SEGMENT_ORDER:
    churn_times = df.loc[df["segment"] == segment, "months_to_churn"].values
    n = len(churn_times)
    # S(t) = proportion still active, i.e. months_to_churn > t
    survival_curves[segment] = np.array([(churn_times > t).sum() / n for t in months])

# Preview: survival at key milestones
milestones = [0, 3, 6, 12, 24, 36, 60]
survival_df = pd.DataFrame(
    {seg: survival_curves[seg][[m for m in milestones]] for seg in SEGMENT_ORDER},
    index=[f"{m} months" for m in milestones],
).T

survival_df.style.format("{:.1%}").background_gradient(cmap="RdYlGn", axis=None)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for segment in SEGMENT_ORDER:
    ax.plot(
        months,
        survival_curves[segment],
        color=SEGMENT_COLOURS[segment],
        linewidth=2.5,
        label=segment,
    )

# Reference lines at 50 % and 25 % survival
for level, ls in [(0.50, "--"), (0.25, ":")]:
    ax.axhline(level, color="grey", linewidth=1, linestyle=ls, alpha=0.6,
               label=f"{level:.0%} survival" if level == 0.50 else None)

ax.set_xlabel("Month Since Acquisition", fontsize=12)
ax.set_ylabel("Fraction of Customers Still Active  S(t)", fontsize=12)
ax.set_title("Cohort Survival Curves by Segment", fontsize=14, fontweight="bold")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_xlim(0, 60)
ax.set_ylim(0, 1.02)
ax.legend(fontsize=10)

# Annotate where each segment crosses 50 % survival
for segment in SEGMENT_ORDER:
    curve = survival_curves[segment]
    crosses = np.where(curve <= 0.5)[0]
    if len(crosses) > 0:
        t50 = months[crosses[0]]
        ax.annotate(
            f"  t½ ≈ {t50} mo",
            xy=(t50, 0.5),
            color=SEGMENT_COLOURS[segment],
            fontsize=9,
            va="center",
        )

plt.tight_layout()
plt.savefig("../reports/figures/survival_curves.png", bbox_inches="tight")
plt.show()

---
## 6 — Outlier Check

We flag customers whose `monthly_spending` falls more than **3 standard deviations** above
their segment mean. The threshold is computed *within each segment* — a R 900/month
`churn_risk` customer is a genuine outlier even though that same value would be unremarkable
in `high_value`.

**Why this matters for CLV modelling:**
- A small number of extreme spenders can dominate the simulated portfolio value if their
  spend level is used to parameterise the Monte Carlo without winsorisation.
- In real data, outliers often represent data-entry errors, one-off bulk orders, or
  fraudulent accounts — all of which should be treated separately.
- Even in this synthetic dataset the Gamma distribution produces occasional extreme draws,
  so it is good practice to know where they live before fitting any model.

In [ ]:
# Compute per-segment z-scores for monthly_spending
segment_stats = df.groupby("segment")["monthly_spending"].agg(["mean", "std"])

df["spend_zscore"] = df.apply(
    lambda row: (
        (row["monthly_spending"] - segment_stats.loc[row["segment"], "mean"])
        / segment_stats.loc[row["segment"], "std"]
    ),
    axis=1,
)

OUTLIER_THRESHOLD = 3.0
outliers = df[df["spend_zscore"] > OUTLIER_THRESHOLD].copy()

print(f"Outliers (spend z-score > {OUTLIER_THRESHOLD}): {len(outliers)} customers"
      f" ({100 * len(outliers) / len(df):.2f}% of total)")

outliers_summary = (
    outliers
    .groupby("segment")
    .agg(
        count=("customer_id", "count"),
        min_spend=("monthly_spending", "min"),
        max_spend=("monthly_spending", "max"),
        mean_spend=("monthly_spending", "mean"),
        max_zscore=("spend_zscore", "max"),
    )
    .loc[[s for s in SEGMENT_ORDER if s in outliers["segment"].unique()]]
    .round(2)
)

outliers_summary.style.format({
    "count": "{:,}",
    "min_spend":  "R {:.2f}",
    "max_spend":  "R {:.2f}",
    "mean_spend": "R {:.2f}",
    "max_zscore": "{:.2f}σ",
})

In [ ]:
# Visualise: scatter plot of all customers, with outliers highlighted
fig, ax = plt.subplots(figsize=(10, 5))

for segment in SEGMENT_ORDER:
    seg_data = df[df["segment"] == segment]
    normal = seg_data[seg_data["spend_zscore"] <= OUTLIER_THRESHOLD]
    extreme = seg_data[seg_data["spend_zscore"] > OUTLIER_THRESHOLD]

    ax.scatter(
        normal["months_to_churn"],
        normal["monthly_spending"],
        color=SEGMENT_COLOURS[segment],
        alpha=0.15,
        s=8,
        label=segment,
        rasterized=True,
    )
    if len(extreme) > 0:
        ax.scatter(
            extreme["months_to_churn"],
            extreme["monthly_spending"],
            color=SEGMENT_COLOURS[segment],
            edgecolors="black",
            linewidths=0.8,
            s=60,
            zorder=5,
            label=f"{segment} outliers (n={len(extreme)})",
        )

ax.set_xlabel("Months to Churn", fontsize=12)
ax.set_ylabel("Monthly Spending (R)", fontsize=12)
ax.set_title(
    f"Spend vs. Churn Tenure — Outliers (z > {OUTLIER_THRESHOLD}σ) highlighted with border",
    fontsize=13, fontweight="bold"
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R {x:,.0f}"))
ax.legend(fontsize=9, markerscale=1.5)

plt.tight_layout()
plt.savefig("../reports/figures/outlier_scatter.png", bbox_inches="tight")
plt.show()

---
## Summary of Key Findings

| Finding | Implication for CLV Modelling |
|---|---|
| Segments are cleanly separated by spend | Segment label alone captures most spend variance — model separately |
| `churn_risk` has infant-mortality Weibull (k < 1) | Early intervention is critical; survival drops steeply in months 0–6 |
| `standard` has memoryless churn (k ≈ 1) | A constant monthly churn-rate assumption is valid for this segment |
| `high_value` has wear-out Weibull (k > 1) | Hazard grows with tenure; long-term retention programmes have diminishing returns |
| < 0.3 % outliers by 3σ rule | Safe to proceed without winsorisation, but worth revisiting on real data |
| Survival crosses 50 % much faster for `churn_risk` | CLV for this segment is dominated by months 1–6 revenue; discount rate matters less |

**Next step:** `02_monte_carlo_clv.ipynb` — fit Weibull parameters per segment and run
Monte Carlo simulations to produce CLV distributions with confidence intervals.